<a href="https://colab.research.google.com/github/maliaheupel/ds2002-fa2/blob/main/04-%20Json/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._ item purchased.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
joined = lines.merge(
    pd.json_normalize(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']

if not unmatched.empty:
    print("unmatched vendor ID:", unmatched['vendor_id'].iloc[0])
    print("units attached:", unmatched['qty'].sum())
else:
    print("all vendors match.")

joined

all vendors match.


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
q3 = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
q3['name'] = q3['name'].fillna('Unknown / Unmatched Vendor')

# used dropna=False or .fillna so that units sold by unmatched vendors (like V-10) remain visible in business sales reporting rather than being silently discarded.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_summary = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
print("quantity per zone")
print(zone_summary)

known_zone_total = joined.dropna(subset=['zone'])['qty'].sum()
unmatched_qty = joined[joined['zone'].isna()]['qty'].sum()

print(f"\nzone A sold the most items with {known_zone_total} items.")
print(f"the total quantity in matched zones: {known_zone_total}")
print(f"the missing 1 item went to vendor V-10, which has no zone metadata.")

quantity per zone
  zone  qty
0    A    7

zone A sold the most items with 7 items.
the total quantity in matched zones: 7
the missing 1 item went to vendor V-10, which has no zone metadata.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_lines = [
    {
        'order': order['order'],
        'vendor_id': order['vendor_id'],
        'item': line.get('item'),
        'qty': line.get('qty')
    }
    for order in ragged_json
    for line in order.get('lines', [{}])
]

len(ragged_lines)

3

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?


a) One row per line item is useful because it's easy to quantify how many items were sold. One row per order would be hard when there's an order that has multiple different items in it.

b) A missing quantity is when the data is lost or we have no idea what the actual number was, but a quantity of zero is a real data point and used when nothing was actually sold. If you fill in missing values with 0, averages would be ruined and brought down.